# Modify and run the data preparation.

https://github.com/karpathy/nanoGPT/blob/master/data/shakespeare/prepare.py

In [46]:
import os
import requests
import tiktoken
import numpy as np

In [47]:
gutenberg_urls = [
    'https://www.gutenberg.org/files/100/100-0.txt',  # Complete Works of Shakespeare
    'https://www.gutenberg.org/files/1342/1342-0.txt',  # Pride and Prejudice by Jane Austen
    'https://www.gutenberg.org/files/11/11-0.txt',      # Alice's Adventures in Wonderland by Lewis Carroll
    'https://www.gutenberg.org/files/84/84-0.txt',      # Frankenstein by Mary Shelley
    'https://www.gutenberg.org/files/1661/1661-0.txt',  # The Adventures of Sherlock Holmes by Arthur Conan Doyle
    'https://www.gutenberg.org/files/2600/2600-0.txt',  # War and Peace by Leo Tolstoy
    'https://www.gutenberg.org/files/98/98-0.txt',      # A Tale of Two Cities by Charles Dickens
    'https://www.gutenberg.org/files/1232/1232-0.txt',  # The Prince by Niccolò Machiavelli
    'https://www.gutenberg.org/files/2701/2701-0.txt',  # Moby Dick by Herman Melville
    'https://www.gutenberg.org/files/74/74-0.txt',      # The Adventures of Tom Sawyer by Mark Twain
    'https://www.gutenberg.org/files/1400/1400-0.txt',  # Great Expectations by Charles Dickens
    'https://www.gutenberg.org/files/46/46-0.txt',      # A Christmas Carol by Charles Dickens
    'https://www.gutenberg.org/files/345/345-0.txt',    # Dracula by Bram Stoker
    'https://www.gutenberg.org/files/768/768-0.txt',    # Wuthering Heights by Emily Brontë
    'https://www.gutenberg.org/files/64317/64317-0.txt',# The Picture of Dorian Gray by Oscar Wilde
    'https://www.gutenberg.org/files/5200/5200-0.txt',  # Metamorphosis by Franz Kafka
    'https://www.gutenberg.org/files/1497/1497-0.txt',  # The Republic by Plato
    'https://www.gutenberg.org/files/8800/8800-0.txt',  # The Divine Comedy by Dante Alighieri
    'https://www.gutenberg.org/files/205/205-0.txt',    # Anna Karenina by Leo Tolstoy
    'https://www.gutenberg.org/files/55/55-0.txt',      # The Wizard of Oz by L. Frank Baum
]

In [48]:
# Directory to save the text files
output_dir = 'gutenberg_texts'
os.makedirs(output_dir, exist_ok=True)

# Combine all downloaded texts into a single file
combined_file_path = 'combined_dataset.txt'

print("Downloading texts from Project Gutenberg...")
with open(combined_file_path, 'w', encoding='utf-8') as combined_file:
    for url in gutenberg_urls:
        file_name = os.path.join(output_dir, url.split('/')[-1])
        
        # Download the text file if it doesn't already exist
        if not os.path.exists(file_name):
            print(f"Downloading {url}...")
            response = requests.get(url)
            with open(file_name, 'w', encoding='utf-8') as f:
                f.write(response.text)
        
        # Append the content to the combined dataset
        with open(file_name, 'r', encoding='utf-8') as f:
            combined_file.write(f.read())
            combined_file.write("\n\n")  # Add spacing between texts

print(f"Combined dataset saved as '{combined_file_path}'.")

Combined dataset saved as 'combined_dataset.txt'.


In [49]:
input_file_path = 'combined_dataset.txt'
# if not os.path.exists(input_file_path):
#     # data_url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
#     data_url = 'https://www.gutenberg.org/files/100/100-0.txt' # Use full 
#     with open(input_file_path, 'w', encoding='utf-8') as f:
#         f.write(requests.get(data_url).text)
    
with open(input_file_path, 'r', encoding='utf-8') as f:
    data = f.read()
    
n = len(data)
print(f'Data length: {n}')
train_data = data[:int(n*0.9)]
val_data = data[int(n*0.9):]

# encode with tiktoken gpt2 bpe
enc = tiktoken.get_encoding("gpt2")
train_ids = enc.encode_ordinary(train_data)
val_ids = enc.encode_ordinary(val_data)
print(f"train has {len(train_ids):,} tokens")
print(f"val has {len(val_ids):,} tokens")

# export to bin files
train_ids = np.array(train_ids, dtype=np.uint16)
val_ids = np.array(val_ids, dtype=np.uint16)
train_ids.tofile(os.path.join(os.getcwd(), 'train.bin'))
val_ids.tofile(os.path.join(os.getcwd(), 'val.bin'))

# train.bin has 301,966 tokens
# val.bin has 36,059 tokens

Data length: 18924637
train has 4,811,059 tokens
val has 509,070 tokens


In [12]:
from datasets import load_dataset
from datasets import DownloadConfig
import tiktoken

ERROR! Session/line number was not unique in database. History logging moved to new session 13


In [13]:
dataset = load_dataset("allenai/c4", "en", split="train", streaming=True)
dataset = dataset.remove_columns(['timestamp', 'url'])

Using the latest cached version of the dataset since allenai/c4 couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'en' at /home/sjiang/.cache/huggingface/datasets/allenai___c4/en/0.0.0/1588ec454efa1a09f29cd18ddd04fe05fc8653a2 (last modified on Fri Apr  4 17:43:33 2025).


In [56]:
from tqdm.notebook import tqdm

In [57]:
enc = tiktoken.get_encoding("gpt2")

current_tokens = 0
all_train_ids = []
all_val_ids = []
iters = 0
for batch in tqdm(iter(dataset)):
    # Split each text into train and validation 
    n = len(batch['text'])
    train_data = batch['text'][:int(n*0.9)]
    val_data = batch['text'][int(n*0.9):]

    train_ids = enc.encode_ordinary(train_data)
    val_ids = enc.encode_ordinary(val_data)

    all_train_ids.append(np.array(train_ids,dtype=np.uint16) )
    all_val_ids.append(np.array(val_ids, dtype=np.uint16))
    
    iters += 1
    current_tokens += len(train_ids)
    
    if current_tokens > 1e8:
        print(f'Used {iters=} batches')
        break

0it [00:00, ?it/s]

Used iters=232966 batches


In [60]:
all_train_ids = np.concatenate(all_train_ids,dtype=np.uint16)
all_val_ids = np.concatenate(all_val_ids,dtype=np.uint16)

In [61]:
# train_ids = np.array(train_ids, dtype=np.uint16)
# val_ids = np.array(val_ids, dtype=np.uint16)
all_train_ids.tofile(os.path.join(os.getcwd(), 'train.bin'))
all_val_ids.tofile(os.path.join(os.getcwd(), 'val.bin'))



In [64]:
enc.decode(all_train_ids[5034:5900])

', because I felt like I’d seen them both six months before.\nIn both cases the trailer revealed the entire structure of the story, key plot twists and expensive action sequences. Throw in a few character deaths for good measure and you’ve got the basis of a significant chunk of what you’ve paid your money for.\nI really did feel cheated by what the studio had wanted me to see in advance.\nSo is this just a modern trend? I checked out trailers for 1981’s Raiders of the Lost Ark, and 1964’s Goldfinger to get a bigger picture.\nBoth revealed some well-known scenes (including Goldfinger’s iconic laser and dialogue), but the plot basics were instead explained by voiceover, rather than any especially huge visual giveaways.\nBesides, the chance of actually seeing these trailers was significantly lessened due to the technology available at the time of release. Which got me thinking further.\n‘If you don’t like it, don’t watch it’, I hear you cry. I wish it were that simple. In the age of the 

In [ ]:
print("Encoding the dataset using GPT-2 BPE...")
enc = tiktoken.get_encoding("gpt2")

# Sample training 

In [17]:
import os
import time
import math
import pickle
from contextlib import nullcontext

import numpy as np
import torch
from model import GPTConfig, GPT

In [18]:
batch_size = 64
block_size = 256 # context of up to 256 previous characters

n_layer = 32
n_head = 6
n_embd = 384
dropout = 0.1


In [19]:
learning_rate = 5e-4 # with baby networks can afford to go a bit higher
max_iters = 2000
lr_decay_iters = 2000 # make equal to max_iters usually
min_lr = 6e-5 # learning_rate / 10 usually
beta1 = 0.9
beta2 = 0.95
weight_decay = 1e-1

warmup_iters = 500

In [20]:
device = 'cuda'
dtype = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16' # 'float32', 'bfloat16', or 'float16', the latter will auto implement a GradScaler
compile = True # use PyTorch 2.0 to compile the model to be faster
torch.manual_seed(1337)
torch.backends.cuda.matmul.allow_tf32 = True # allow tf32 on matmul
torch.backends.cudnn.allow_tf32 = True # allow tf32 on cudnn
device_type = 'cuda' if 'cuda' in device else 'cpu' # for later use in torch.autocast
# note: float16 data type will automatically use a GradScaler
# ptdtype = {'float32': torch.float32, 'bfloat16': torch.bfloat16, 'float16': torch.float16}[dtype]
# ctx = nullcontext() if device_type == 'cpu' else torch.amp.autocast(device_type=device_type, dtype=ptdtype)

In [31]:
# Simple data grabbing thing
def get_batch(split):
    # We recreate np.memmap every batch to avoid a memory leak, as per
    # https://stackoverflow.com/questions/45132940/numpy-memmap-memory-usage-want-to-iterate-once/61472122#61472122
    if split == 'train':
        data = np.memmap('train.bin', dtype=np.uint16, mode='r')
    else:
        data = np.memmap('val.bin', dtype=np.uint16, mode='r')
    ix = torch.randint(len(data) - block_size, (batch_size,))
    print(ix)
    x = torch.stack([torch.from_numpy((data[i:i+block_size]).astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy((data[i+1:i+1+block_size]).astype(np.int64)) for i in ix])
    if device_type == 'cuda':
        # pin arrays x,y, which allows us to move them to GPU asynchronously (non_blocking=True)
        x, y = x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)
    else:
        x, y = x.to(device), y.to(device)
    return x, y

In [ ]:
enc.decode_batch(get_batch('train')[0].cpu().tolist())

In [7]:
model_args = dict(n_layer=n_layer, n_head=n_head, n_embd=n_embd, block_size=block_size,
                  bias=False, vocab_size=None, dropout=dropout) # start with model_args from command line

In [8]:
# init a new model from scratch
print("Initializing a new model from scratch")
# determine the vocab size we'll use for from-scratch training
model_args['vocab_size'] = 50304
gptconf = GPTConfig(**model_args)
model = GPT(gptconf)

Initializing a new model from scratch
no weight tying
number of parameters: 95.28M


In [9]:
# This seems a lot more complicated... 
# optimizer = model.configure_optimizers(weight_decay, learning_rate, (beta1, beta2), device_type)
optimizer = torch.optim.AdamW(model.parameters(), 
                              lr=learning_rate, betas=(beta1, beta2))


In [10]:
optimizer

AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.95)
    capturable: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0005
    maximize: False
    weight_decay: 0.01
)

In [11]:
unoptimized_model = model
model = model.to('cuda')
# model = torch.compile(model) # requires PyTorch 2.0

In [12]:
def get_lr(it):
    # 1) linear warmup for warmup_iters steps
    if it < warmup_iters:
        return learning_rate * (it + 1) / (warmup_iters + 1)
    # 2) if it > lr_decay_iters, return min learning rate
    if it > lr_decay_iters:
        return min_lr
    # 3) in between, use cosine decay down to min learning rate
    decay_ratio = (it - warmup_iters) / (lr_decay_iters - warmup_iters)
    assert 0 <= decay_ratio <= 1
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio)) # coeff ranges 0..1
    return min_lr + coeff * (learning_rate - min_lr)

In [13]:
X, Y = get_batch('train') # fetch the very first batch

In [14]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out


In [15]:
local_iter_num = 0 # number of iterations in the lifetime of this process
iter_num = 0
eval_interval = 250
eval_iters = 200
gradient_accumulation_steps = 1
grad_clip = 1.0
all_loss = []
while True:
    lr = get_lr(iter_num)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr

    if iter_num % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter_num}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    for micro_step in range(1):
        logits, loss = model(X, Y)
        loss = loss / gradient_accumulation_steps # scale the loss to account for gradient accumulation
        # immediately async prefetch next batch while model is doing the forward pass on the GPU
        X, Y = get_batch('train')
        all_loss.append(loss.item())
        # backward pass, with gradient scaling if training in fp16
        loss.backward()
    
    # if grad_clip != 0.0:
    #     torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        # step the optimizer and scaler if training in fp16
    
    optimizer.step()  # Directly call step on the optimizer
    optimizer.zero_grad(set_to_none=True)

    iter_num += 1
    local_iter_num += 1

    # termination conditions
    if iter_num > max_iters:
        break


step 0: train loss 10.8877, val loss 10.8841
step 250: train loss 4.8793, val loss 5.2543
step 500: train loss 3.9993, val loss 4.7092
step 750: train loss 3.5195, val loss 4.4913
step 1000: train loss 3.2019, val loss 4.3679
step 1250: train loss 2.9578, val loss 4.3710
step 1500: train loss 2.7226, val loss 4.3776
step 1750: train loss 2.5430, val loss 4.4128
step 2000: train loss 2.4325, val loss 4.4465


In [16]:
model.eval()
enc = tiktoken.get_encoding("gpt2")
encode = lambda s: enc.encode(s, allowed_special={"<|endoftext|>"})
decode = lambda l: enc.decode(l)
start = '\n'
if start.startswith('FILE:'):
    with open(start[5:], 'r', encoding='utf-8') as f:
        start = f.read()
start_ids = encode(start)
x = (torch.tensor(start_ids, dtype=torch.long, device=device)[None, ...])
y = model.generate(x, 100, temperature=0.8)

In [17]:
print(decode(y[0].tolist()))



KING.
By this day, I will.

[_Exeunt._]

SCENE V. The same. A Room in the Palace

Enter Lord Chamberlain, the Duke of York and Sir Nicholas Vaux.

CHAMBERLAIN.
Good morrow, gracious sovereign. I have heard you say,
As many of the last year, I have forgot,
Have played the blow on the head of York, and given it
To a


In [1]:
import torch

In [5]:
checkpoint = torch.load(f'gpt-save-parallel-2/model_checkpoint_{0}_iter_num={2200}', weights_only=False,map_location=torch.device('cpu'))


In [ ]:
checkpoint